# Calenda — Qwen3-0.6B LoRA 학습 (Kaggle · T4 단일 GPU)

단일 베이스 **Qwen/Qwen3-0.6B** · **completion-only 손실 ON**.

**사전 설정 (노트북 실행 전)**
- Notebook settings → Accelerator: **GPU T4 x1** (또는 T4 x2 중 하나 선택)
- Add-ons → Secrets: `HF_TOKEN` 등록 (HuggingFace write token)
- Settings → Internet: **On**

위→아래 순서 실행.

## 0. 환경 초기화

In [ ]:
import os

# Kaggle Secrets에서 HF_TOKEN 로드
try:
    from kaggle_secrets import UserSecretsClient
    _hf_token = UserSecretsClient().get_secret('HF_TOKEN') or ''
except Exception:
    _hf_token = ''

os.environ['HF_TOKEN'] = _hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = _hf_token
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['WANDB_DISABLED'] = 'true'
print('HF_TOKEN:', '있음' if _hf_token else '없음 (LoRA HF 업로드 스킵됨)')

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. repo clone + 버전 배너

In [ ]:
import os, yaml, json, subprocess
from collections import Counter

WORK = '/kaggle/working'
REPO = f'{WORK}/Calenda'

%cd {WORK}
if not os.path.exists(REPO):
    !git clone https://github.com/sooryong/Calenda.git
else:
    !cd {REPO} && git fetch origin && git reset --hard origin/main
%cd {REPO}

_c = yaml.safe_load(open('configs/train_qwen3_0_6b.yaml', encoding='utf-8'))
def _stat(p):
    rows = [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
    c = Counter(str(r['gold'].get('schedule_status')) for r in rows)
    return len(rows), c.get('yes', 0), c.get('pending', 0), c.get('no', 0)
_tt, _ty, _tp, _tn = _stat('data/processed/train.jsonl')
_gt, _gy, _gp, _gn = _stat('data/eval/golden.jsonl')
_head = subprocess.run('git log --oneline -1', shell=True, capture_output=True, text=True).stdout.strip()
print('=' * 62)
print(f"  라운드          : {_c['run_name']}  ·  epochs={_c['num_train_epochs']}")
print(f"  completion-only : {_c.get('completion_only_loss', False)}")
print(f"  커밋            : {_head[:60]}")
print(f"  학습셋          : train {_tt}   yes {_ty} / pending {_tp} / no {_tn}")
print(f"  평가셋          : golden {_gt}   yes {_gy} / pending {_gp} / no {_gn}")
print('=' * 62)

## 3. 의존성 설치

`cuda: True` 확인 후 진행.

In [ ]:
!pip install -q -e .[train]
!pip uninstall torchao -y -q 2>/dev/null || true
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'ngpu:', torch.cuda.device_count())

## 4. 데이터 확인

In [ ]:
import os
for p in ['data/processed/train.jsonl', 'data/eval/golden.jsonl']:
    n = sum(1 for l in open(p, encoding='utf-8') if l.strip())
    print(f'{p}: {n} records')
_val = 'data/processed/val.jsonl'
if os.path.isfile(_val):
    print(f'{_val}: {sum(1 for l in open(_val, encoding="utf-8") if l.strip())} records')
else:
    print(f'{_val}: 없음')

## 5. 사전점검 — 학습 렌더 == 추론 프롬프트 + gold JSON

`OK` 확인 후 학습 진행.

In [ ]:
import os, yaml, json
from transformers import AutoTokenizer

CONFIG = 'configs/train_qwen3_0_6b.yaml'
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
print(f"라운드 {_cfg['run_name']} | base {_mcfg['hf_id']}")
print(f"output_dir {_cfg['output_dir']} | 유효배치 {_cfg['per_device_train_batch_size'] * _cfg['gradient_accumulation_steps']}")

_tok = AutoTokenizer.from_pretrained(
    _mcfg['hf_id'],
    trust_remote_code=_mcfg.get('trust_remote_code', False),
    token=False,
)
_sys  = _mcfg['system_prompt']
_user = '<채널: KakaoTalk>\n<메시지>\n내일 3시 회의\n</메시지>'
_gold = json.dumps({'schedule_status': 'no', 'events': []}, ensure_ascii=False)
_full = [{'role': 'system', 'content': _sys}, {'role': 'user', 'content': _user}, {'role': 'assistant', 'content': _gold}]
_pre  = [{'role': 'system', 'content': _sys}, {'role': 'user', 'content': _user}]
_train_str = _tok.apply_chat_template(_full, tokenize=False, add_generation_prompt=False)
_extra = {'enable_thinking': False} if 'enable_thinking' in (getattr(_tok, 'chat_template', None) or '') else {}
_infer_str = _tok.apply_chat_template(_pre, tokenize=False, add_generation_prompt=True, **_extra)
_aligned   = _train_str.startswith(_infer_str)
_json_next = (_train_str[len(_infer_str):] if _aligned else '').lstrip().startswith('{')
_rt_ok     = '<|im_start|>assistant\n' in _train_str
print(f'정렬: {_aligned} | JSON바로시작: {_json_next} | response_template: {_rt_ok}',
      '→ OK' if (_aligned and _json_next and _rt_ok) else '→ FAIL')
assert _aligned and _json_next, '학습/추론 프롬프트 정렬 깨짐'

## 6. 학습

In [ ]:
print(f"학습 시작 → {_cfg['run_name']} · 단일 GPU")
!python scripts/train_lora.py --config {CONFIG}

## 7. 평가 (LoRA merge → 골든셋 평가)

In [ ]:
import yaml, os
CONFIG = 'configs/train_qwen3_0_6b.yaml'
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))

BASE       = _mcfg['hf_id']
LORA_DIR   = _cfg['output_dir']
NAME       = os.path.basename(LORA_DIR)
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON  = f'logs/eval_{NAME}.json'
GOLDEN     = _cfg.get('eval_golden', 'data/eval/golden.jsonl')

print(f'lora: {LORA_DIR} → merged: {MERGED_DIR}')
!python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON} --model_config {_cfg['model_config']}

print(f'\n--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 8. 결과 저장

Kaggle Output 탭에서 파일을 다운로드할 수 있습니다.
아래 셀은 `/kaggle/working/` 루트에 다운로드용 파일을 복사합니다.

In [ ]:
import shutil, os

OUT = '/kaggle/working'

# 8-1. eval JSON 복사
shutil.copy(EVAL_JSON, f'{OUT}/eval_{NAME}.json')
print(f'eval → {OUT}/eval_{NAME}.json')

# 8-2. failures 복사
FAILURES_SRC = 'data/failures/round_latest.jsonl'
FAILURES_DST = f'{OUT}/failures_{NAME.split("-")[0]}.jsonl'
if os.path.isfile(FAILURES_SRC):
    shutil.copy(FAILURES_SRC, FAILURES_DST)
    print(f'failures → {FAILURES_DST} ({sum(1 for l in open(FAILURES_DST) if l.strip())}건)')
else:
    print('failures 없음')

# 8-3. LoRA zip
ZIP = f'{OUT}/lora_{NAME}.zip'
shutil.make_archive(ZIP[:-4], 'zip', LORA_DIR)
print(f'lora zip → {ZIP} ({os.path.getsize(ZIP) // 1024 // 1024} MB)')

print('\nKaggle Output 탭에서 위 파일들을 다운로드하세요.')

## 9. (선택) HuggingFace에 LoRA 어댑터 백업

HF_TOKEN Secret이 등록된 경우에만 실행.

In [ ]:
import os
if os.environ.get('HF_TOKEN'):
    REPO_ID = f'sooryong9885/Calenda-Qwen3-0.6B'
    !python -c "
from huggingface_hub import HfApi
import os
api = HfApi(token=os.environ['HF_TOKEN'])
api.upload_folder(folder_path='{LORA_DIR}', repo_id='{REPO_ID}', repo_type='model', commit_message='Upload {NAME}')
print('HF 업로드 완료: {REPO_ID}')
"
else:
    print('HF_TOKEN 없음 — 스킵')